# 02 · pandas 上：把数据装进表格

**时间预算**：3 小时。前 1 小时 40 分钟跟着跑代码，剩下做练习。

上节课结尾留了个问题：numpy 数组能算，但**没有名字**——你只能说"第 3 列"，不能说"票价"。
今天解决它。`DataFrame` 就是**带标签的表格**：行有索引标签，列有列名。

学完你应该能回答三个问题：

1. `df[...]`、`df.loc[...]`、`df.iloc[...]` 三者到底有什么区别？
2. 缺失值该丢掉、该填、还是该留着？
3. 为什么我明明写了赋值语句，pandas 不报错、值却没变？

## 1. 读进来，先认清它是什么

In [ ]:
from statistics import mean

import pandas as pd

pd.set_option("display.max_columns", 60)  # 别让列显示成省略号
print("pandas 版本:", pd.__version__)

df = pd.read_csv("../data/raw/titanic.csv")  # 路径相对 notebook 所在目录
print(type(df))
df.head()

### DataFrame 的三个零件

| 零件 | 是什么 | 怎么取 |
| :--- | :--- | :--- |
| `index` | 行的标签 | `df.index` |
| `columns` | 列的名字 | `df.columns` |
| `values` | 底层的数据，**就是一个 numpy 数组** | `df.values` |

第三行很重要：**pandas 是在 numpy 上包了一层标签**。
所以上节课学的广播、布尔索引、axis，在 pandas 里全部适用——只是多了名字可以叫。

In [ ]:
print("行索引:", df.index[:5].tolist(), "...")
print("列名  :", df.columns.tolist())
print()
print("底层数据是  :", type(df.values).__name__, df.values.shape)
print("单列取出来是:", type(df["Age"]).__name__, "  <- 一维，叫 Series")
print("多列取出来是:", type(df[["Age", "Fare"]]).__name__, "  <- 二维，还是 DataFrame")

## 2. 看数据的五个动作

**这节课最重要的部分。** 面对任何一份陌生数据（别人给的 CSV、数据库导出的表），
按这五步走一遍，你就能对它有基本判断。这不是我编的，是数据分析的通用套路。

1. **看形状**：多少行多少列 → `df.shape`
2. **看类型和完整性**：每列是什么类型、有没有空 → `df.info()`
3. **看数值分布**：范围、均值、中位数、有没有离谱的值 → `df.describe()`
4. **看类别取值**：每种类别各有多少 → `value_counts()`
5. **看缺失**：哪列缺、缺多少 → `isna().sum()`

下面逐个跑一遍。**每看一格，停 3 秒想想"这条信息告诉我什么"**，别只是往下点。

In [ ]:
# 第一问：形状
print("形状（行, 列）:", df.shape)
print("列名:", df.columns.tolist())
print()
print("前 3 行:")
print(df.head(3).to_string())
print()
print("后 2 行:")
print(df.tail(2).to_string())

In [ ]:
# 随机抽 5 行 —— 看中间的数据比 head 更有信息量，head 容易让人以为数据很整齐
df.sample(5, random_state=42)

In [ ]:
# 第二问：类型 + 非空数量。这是最快了解一份数据的方法
df.info()

### 注意 dtype 那一列

- `int64` / `float64` 是数值；
- `str` 是文本。**在旧版 pandas 里这里显示的是 `object`**，pandas 3.0 换成了更明确的 `str` 类型；
- `Age` 是 `float64` 而不是 `int64`——**年龄怎么会是小数？因为有空值**。
  numpy 的整数数组装不下 `NaN`，所以只要一列有缺失的整数，pandas 就把它变成浮点。
  这类"从类型反推数据问题"的推理，是做数据的基本功。

In [ ]:
# 第三问：数值列的分布
df.describe().round(2)

重点看两个地方：

- `Age` 的 count 是 714，而总行数是 891 —— **这直接告诉你少了 177 个年龄**；
- `Fare` 的最小值是 0（有免费票？），最大值 512，而中位数只有 14.45 —— 
  **均值 32 被少数高价票拉高了**。这种情况下用中位数比均值更能代表"典型乘客"。

In [ ]:
# 第四问：类别列的取值分布
for col in ["Survived", "Pclass", "Sex", "Embarked"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().to_string())
    print()

# normalize=True 给出的是比例而不是个数
print("整体生存率:", round(df["Survived"].mean(), 3))

In [ ]:
# 第五问：缺失值。把数量和比例并排放，一眼就能判断严重程度
summary = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例%": (df.isna().mean() * 100).round(1),
})
summary[summary["缺失数量"] > 0].sort_values("缺失数量", ascending=False)

## 3. 缺失值：三种策略，怎么选

看到 `Cabin` 缺 77%、`Age` 缺 20%、`Embarked` 缺 2 行，你的第一反应是什么？

很多人的答案是"全丢掉"或者"用均值填上"。**两个都错**，因为处理方式取决于三件事：

| 策略 | 什么时候用 | 代价 |
| :--- | :--- | :--- |
| **丢掉**（`dropna`） | 缺得极少（比如 2 行），丢掉不影响代表性 | 样本变少 |
| **填上**（`fillna`） | 缺得不多，且这一列必须要有值才能算 | 填的是猜的值，会削弱这一列的真实信息 |
| **标记**（新增一列记录"这里缺了"） | 缺得很多（比如 77%） | 需要多一列，但保住了「缺失」本身的信息 |

**为什么第三种常常比硬填更好**：如果某一列缺失本身就说明了一些事，硬填等于把这个信号抹掉。
比如"没有舱位记录"的乘客，很可能本来就没买有记录的舱位——这是信息，不是噪音。

下面三种各演示一次，你自己看结果。

In [ ]:
# 策略 1：丢掉 —— Embarked 只缺 2 行，占比 0.2%，丢掉几乎无损
df_drop = df.dropna(subset=["Embarked"])
print(f"丢掉 Embarked 缺失的行: {df.shape} -> {df_drop.shape}")

In [ ]:
# 策略 2：填 —— Age 用中位数。为什么不是均值？因为均值被极端值拉偏了
age_median = df["Age"].median()
print(f"Age 中位数: {age_median:.1f}   均值: {df['Age'].mean():.1f}")

df_filled = df.copy()
df_filled["Age"] = df_filled["Age"].fillna(age_median)
print("填完之后还有缺失吗:", df_filled["Age"].isna().any())
print("填之前描述:", df["Age"].describe().round(2).to_dict())
print("填之后描述:", df_filled["Age"].describe().round(2).to_dict())

In [ ]:
# 策略 3：标记 —— Cabin 缺 77%，硬填等于编造数据，不如把「缺失」变成一个新特征
df_filled["CabinMissing"] = df_filled["Cabin"].isna()
print("CabinMissing 的分布:")
print(df_filled["CabinMissing"].value_counts().to_string())
print()
print("「有没有舱位记录」和生存率的关系:")
print(df_filled.groupby("CabinMissing")["Survived"].mean().round(3).to_string())
print()
print("  ^ groupby 是下节课的内容，这里先提前看一眼效果")

最后一个输出如果显示"有记录的人生存率明显更高"——**光凭这个缺失模式就预测出了不少信息**。
这就是为什么我说"缺失"本身常常是数据，不是垃圾。

> 顺便说一句：`df_filled["Age"] = ...` 这种**整列赋值**是安全的；
> 第 5 节会讲哪种写法是危险的。

## 4. 选数据：`[]`、`.loc`、`.iloc`

这是 pandas 最容易搞混的地方。先把三种括号的用途分清楚：

In [ ]:
# [] 的三种用法
print("1) 一列       :", type(df["Age"]).__name__, "  <- Series")
print("2) 多列       :", type(df[["Age", "Fare"]]).__name__, "  <- DataFrame")

mask = df["Age"] > 60
print("3) 布尔筛选用 : mask.sum() =", mask.sum(), "人；df[mask].shape =", df[mask].shape)

### 为什么必须区分 `.loc` 和 `.iloc`

- `.loc[行标签, 列名]` —— **label**，按名字找；
- `.iloc[行位置, 列位置]` —— **integer position**，按排第几找。

不筛选的时候，两者结果常常一样（因为默认索引正好是 0,1,2...）。
**但一旦筛选过，索引就不再连续**——这时候它们会给出完全不同的结果。看下面。

In [ ]:
old = df[df["Age"] > 60]
print("筛选后前 6 个索引标签:", old.index[:6].tolist())
print("  ^ 索引不再是 0,1,2,3 连着的了——这就是必须区分 loc 和 iloc 的原因")
print()
print("iloc[0] 第 0 个位置的人    :", old.iloc[0]["Name"])
print("loc[33] 索引标签为 33 的人 :", old.loc[33, "Name"])
print()
print("注意 loc[0] 会直接报错，因为筛选后根本没有 0 这个标签：")

In [ ]:
try:
    old.loc[0, "Name"]
except KeyError as exc:
    print("KeyError:", exc)
    print("  -> 报错信息说得很清楚：0 不在索引里。看到 KeyError 就先检查索引")

In [ ]:
# .loc[行标签, 列名]
print(df.loc[0, "Name"])
print()
print(df.loc[0:2, ["Name", "Age"]].to_string())

# .iloc[行位置, 列位置]
print()
print(df.iloc[0:3, 0:4].to_string())

### 一个必须记住的差异：切片含不含终点

`loc` 的切片**包含**终点，`iloc` 的切片**不包含**终点。
这和 numpy 的切片规则一致（numpy 不含终点），但和 Python 列表的直觉不一样。

In [ ]:
print("df.loc[0:2]  取到", df.loc[0:2].shape[0], "行  <- 含终点")
print("df.iloc[0:2] 取到", df.iloc[0:2].shape[0], "行  <- 不含终点")
print()
print("记忆：loc 是「从 0 到 2 都要」，iloc 是「从 0 数 2 个」")

## 5. pandas 3.0 的坑：链式赋值静默失效

这是**这周最容易浪费你一小时的问题**。很多人会这么写：

```python
df[df["Sex"] == "female"]["Fare"] = 0     # 看起来没问题，其实完全没生效
```

它的意思是"先筛出女性的行，再改其中的 Fare 列"。听起来完全合理。
但你实际改的是**一个临时副本**，原来的 `df` 一点没变，**而且不报错**。

pandas 3.0 默认开启了 Copy-on-Write，这种情况会打印一条警告（旧版本是静默的）。
**警告不是错误，程序照跑**——所以它极其容易被忽略。看下面的演示。

In [ ]:
broken = df.copy()

# 错误写法：链式赋值
broken[broken["Sex"] == "female"]["Fare"] = 0

print("改完之后，女性票价的最大值:", broken.loc[broken["Sex"] == "female", "Fare"].max())
print("  ^ 不是 0，说明上面那行赋值根本没生效")
print("同一格下面会打印一条 ChainedAssignmentError 警告，仔细读它说了什么")

In [ ]:
# 正确写法：用 .loc 一次说清「哪些行、哪一列」
fixed = df.copy()
fixed.loc[fixed["Sex"] == "female", "Fare"] = 0

print("正确写法之后，女性票价的最大值:", fixed.loc[fixed["Sex"] == "female", "Fare"].max())
print("  ^ 变成 0 了")

### 一句话规则

> **要改值，就用 `df.loc[行的条件, "列名"] = 值`，一次把行列都指名道姓写在同一个中括号里。**

只要出现**两个连续的中括号**（像 `df[...][...] = ...`），就是危险信号。

顺便回头想想上节课的内容：numpy 里"切片给视图、花式索引给副本"也是同一类问题——
**你拿到的是原件还是复印件**，决定了改它有没有用。这个主题会跟你一辈子。

## 6. 筛选、排序、加列

这节是今后每天都会用到的三个基本操作。

In [ ]:
# 单条件筛选
female = df[df["Sex"] == "female"]
male = df[df["Sex"] == "male"]
print(f"女性 {len(female)} 人生存率 {female['Survived'].mean():.3f}")
print(f"男性 {len(male)} 人生存率 {male['Survived'].mean():.3f}")
print()

# 多条件：每个条件都要括号，用 & | ~，不能用 and / or / not
rich_women = df[(df["Sex"] == "female") & (df["Fare"] > 50)]
print(f"票价超过 50 的女性 {len(rich_women)} 人，生存率 {rich_women['Survived'].mean():.3f}")
print()

# isin 和 between 这两个便捷方法
print("1 等舱或 2 等舱的人数:", df["Pclass"].isin([1, 2]).sum())
print("票价在 10~30 之间的人数:", df["Fare"].between(10, 30).sum())
print("非女性的行数:", (~(df["Sex"] == "female")).sum(), "  <- ~ 是取反")

In [ ]:
# 排序
top5 = df.sort_values("Fare", ascending=False).head(5)
print("票价最高的 5 位:")
print(top5[["Name", "Pclass", "Fare", "Survived"]].to_string(index=False))
print()

# 按多列排序：先按舱位升序，同舱位再按票价降序
print("舱位内票价最高的几位:")
sorted_df = df.sort_values(["Pclass", "Fare"], ascending=[True, False])
print(sorted_df[["Pclass", "Fare", "Name"]].head(6).to_string(index=False))

In [ ]:
# 加一列：新建整列是安全的，可以直接赋值
df2 = df.copy()
df2["FamilySize"] = df2["SibSp"] + df2["Parch"] + 1   # 加上自己

print("家庭人数分布:")
print(df2["FamilySize"].value_counts().sort_index().to_string())
print()
print("检查一下有没有算错，第 0 行的 SibSp/Parch/FamilySize:")
print(df2.loc[0, ["SibSp", "Parch", "FamilySize"]].to_dict())

## 7. 练习（约 1 小时）

做完把每格的输出发我，**第 1 题和第 6 题要写文字说明**，别只贴代码。

### 练习 1（观察力，文字题）

把第 2 节那五个动作在 `df` 上跑一遍，然后写下**你认为最值得注意的三条发现**。
每条用一句话，格式是"什么现象 + 可能意味着什么"。

> 这题没有标准答案，我看的是你能不能从数字里读出问题。
> 提示：注意 `Fare` 的中位数和均值差多少、`Age` 缺多少、各列的取值范围有没有不合理的。

In [ ]:
# 你的代码
import pandas as pd
df = pd.read_csv("../data/raw/titanic.csv")

# 看形状：多少行多少列 → df.shape
print("数据形状：",df.shape)
# 看类型和完整性：每列是什么类型、有没有空 → df.info()
print(df[["Pclass","Name"]].info())
print(df.info())
print("==================================")
# 看数值分布：范围、均值、中位数、有没有离谱的值 → df.describe()
print(df.describe().round(2))
print(df[df["Fare"] == 0])
print("==================================")
# 看类别取值：每种类别各有多少 → value_counts()
print(df["Pclass"].value_counts().to_string())
# 看缺失：哪列缺、缺多少 → isna().sum()
print(df.isna().sum().to_string())
# 我的三条发现：
# 1.发现年龄的数据类型为float，年龄应该为整数才对，信息是否有空值
# 2.cabin数据大量缺失是因为什么，发现缺失这个数据的人的生存率较低
# 3.fare中有值为0，票价不应该为0，这 15 个人全部从 S 港上船（全船只有 72% 是 S），15 个人里只活了 1 个（全船 38%），而且一半以上年龄缺失

### 练习 2（排序）

找出票价最低的 5 位乘客，只显示姓名、舱位、票价、是否存活。

In [ ]:
# 你的代码
top5 = df.sort_values("Fare").head(5)
print(top5[["Name","Pclass","Fare","Survived"]].to_string(index=False))
# 提示：sort_values 默认就是升序，别忘了 to_string(inde=False) 让输出好看点

### 练习 3（筛选 + 聚合）

分别算出 1 等舱、2 等舱、3 等舱乘客的生存率。

用三次筛选加三次 `.mean()` 就行（下节课学的 `groupby` 能一行搞定，今天先笨办法）。

In [ ]:
# 你的代码
one = df[df["Pclass"] == 1]
two = df[df["Pclass"] == 2]
tree = df[df["Pclass"] == 3]
print(f"""
一等舱的人数为:{len(one)},生存率为：{one["Survived"].mean():.2f}
二等舱的人数为:{len(two)},生存率为：{two["Survived"].mean():.2f}
三等舱的人数为:{len(tree)},生存率为：{tree["Survived"].mean():.2f}
""")

print(df.groupby("Pclass")["Survived"].mean().round(2).to_string())

# 期望：三个生存率，应该能看出明显的等级差异

### 练习 4（多条件）

找出同时满足这三个条件的乘客有多少人：
**男性**、**年龄小于 15 岁**、**活下来了**。

注意年龄有缺失值，想想缺失的行会不会被算进去。

In [ ]:
# 你的代码
summary = df [(df["Sex"] == "male") & (df["Age"] < 15) & (df["Survived"] == 1)]
print(f"存活下来的人共有{len(summary)}人")
# 想一想：df["Age"] < 15 对缺失值会返回什么？它会被算进结果吗？

### 练习 5（缺失值决策，文字 + 代码）

针对这份数据的三个缺失列，分别写出你的处理方案**和理由**：

- `Cabin` 缺 687 行（77%）
- `Age` 缺 177 行（20%）
- `Embarked` 缺 2 行（0.2%）

用代码实现你的方案。理由写一句话就行，但要说出**为什么不是另外两种**。

In [ ]:
# 你的代码 + 理由
# Cabin     -> 方案：从新标记            理由：缺失的太多了，重新标记一个属性
mark = df.copy()
mark["CabinMissing"] = df["Cabin"].isna()
print(mark["CabinMissing"].value_counts().to_string())
# Age       -> 方案：fillna            理由：用中位数或均值填充
fill_df = df.copy()
median = fill_df["Age"].median()
mean = fill_df["Age"].mean()
print(f"均值：{mean:.2f}  中位数：{median}")
fill_df["Age"] = fill_df["Age"].fillna(median)
print(fill_df["Age"].isna().sum())
# Embarked  -> 方案：dropna            理由：就两条数据直接删除，影响不大
drop_df = df.dropna(subset=["Embarked"])
print(drop_df["Embarked"].isna().sum())

### 练习 6（进阶，考验理解）

下面两句话的结果一样吗？先猜，再跑代码验证，然后用文字解释为什么。

```python
df.loc[0:2, ["Name", "Age"]]
df.iloc[0:2, [0, 1]]
```

再想想：如果我把 `df` 换成"筛选过、索引不连续"的 `old`，结论会变吗？

In [ ]:
# 你的代码 + 结论
print(df.loc[0:2, ["Name", "Age"]])
print(df.iloc[0:2, [0, 1]])
# 输出结果并不相同


old = df[df["Age"] > 60]
print(old.loc[33:100,["Name","Age"]])
print(old.iloc[0:2,[3,5]])
# 我的结论：
# 如果把 df 换成"筛选过、索引不连续"的 old，结论会变
# loc是按标签查看  iloc是按位置查找

## 8. 总结卡片

四句话，抄进你自己的笔记：

1. **看数据五问**：形状 → 类型与完整性 → 数值分布 → 类别取值 → 缺失情况。面对陌生数据的第一反应就是这五步。
2. **选数据**：`df["列"]` 给 Series，`df[["列1","列2"]]` 给 DataFrame；`.loc` 按标签、`.iloc` 按位置，`loc` 切片含终点、`iloc` 不含。
3. **改数据**：永远用 `df.loc[行的条件, "列名"] = 值`。出现连续两个中括号就是危险信号。
4. **缺失值**：丢掉 / 填上 / 标记，取决于"缺多少"和"缺失本身是不是信息"。缺 77% 的列硬填，等于编数据。

---

### 下一课预告

今天全都在"看"数据，还没真正"算"过。刚才演示里偷偷用过一次 `groupby`，
它能一行算出"1/2/3 等舱各自的生存率"——而今天你要写三遍筛选。

下节课：`groupby` / `merge` / `pivot_table`，也就是把数据**从行变成表**的三种手法。

带着一个问题来：如果两张表都记录了一部电影，一张记评分、一张记票房，怎么把它们合成一张？

## 9. 卡住时的提示（先别看）

**练习 1**：三个方向可以看——① 数值列里"不可能是这个值"的东西（票价 0？年龄 0.42？）；
② 中位数和均值差得远的列（说明有极端值）；③ 缺失集中在哪几列，能不能看出规律。

**练习 2**：`sort_values` 不写 `ascending=False` 就是升序，正好是你要的。
选出需要的列用 `[["列1", "列2"]]`（注意两层中括号）。

**练习 3**：三次筛选，每次 `df[df["Pclass"] == n]["Survived"].mean()`。
如果想少写点代码，可以试试把三个筛选放进字典再循环——但先写笨的，能跑通更重要。

**练习 4**：三个条件用 `&` 连起来，每个都要括号。
另外先跑一句 `print((df["Age"] < 15).sum())` 看看结果，再想想 `Age` 是 `NaN` 的那些行走到了哪里。

**练习 5**：没有标准答案，我要的是理由。三个方向：
丢（缺得极少）、填（缺得不多、必须要有值）、标记（缺太多，硬填等于造假）。
注意 `Cabin` 那题——如果你选择"填"，你打算填什么？填得进去吗？

**练习 6**：关键在"两个切片含不含终点"和"列用名字还是用序号"。
`loc[0:2]` 是 3 行，`iloc[0:2]` 是 2 行。把 `df` 换成 `old` 之后，
`loc[0:2]` 会因为找不到标签 0 而报错——先自己跑出来看看报什么。